# Notebook 18: Medicare Advantage Plan Exits, Michigan (2026 → 2027)

**Question:** How many Michigan Medicare Advantage members are in plans that won't continue in 2027, and where are they?

**Data (CMS, public):**
- 2027 Part C&D Plan Crosswalk: how each 2026 plan maps to 2027 (renews, merges, shrinks, or ends)
- Monthly Enrollment by Contract/Plan/State/County, October 2026: members per plan per county

**Operator angle:** When members lose a plan, DME rentals in progress and prior authorizations have to be re-established with a new payer.

**Rules:** Own numbers only, built from the raw files. Definitions taken from the CMS file documentation, not assumed. Small counts reported as CMS publishes them.

In [0]:
import urllib.request, zipfile, os

VOL = "/Volumes/workspace/default/raw_data/nb18"
os.makedirs(VOL, exist_ok=True)
FILES = {
    "crosswalk_2027.zip": "https://www.cms.gov/files/zip/plan-crosswalk-2027.zip",
    "enrollment_cpsc_2026_10.zip": "https://www.cms.gov/files/zip/monthly-enrollment-cpsc-october-2026.zip",
}

for name, url in FILES.items():
    path = f"{VOL}/{name}"
    if not os.path.exists(path):
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(path, "wb") as f:
            f.write(r.read())
    with zipfile.ZipFile(path) as z:
        z.extractall(f"{VOL}/{name[:-4]}")

for root, _, files in os.walk(VOL):
    for fn in files:
        if not fn.endswith(".zip"):
            p = os.path.join(root, fn)
            print(f"{os.path.getsize(p)/1e6:8.1f} MB  {p.replace(VOL + '/', '')}")

In [0]:
def head(path, n=5):
    with open(path, encoding="latin-1") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            print(line.rstrip()[:300])

E = f"{VOL}/enrollment_cpsc_2026_10/CPSC_Enrollment_2026_10"
print("=== Crosswalk (first 5 lines) ===");   head(f"{VOL}/crosswalk_2027/PlanCrosswalk2027_10012026.txt")
print("\n=== Enrollment (first 5 lines) ===");  head(f"{E}/CPSC_Enrollment_Info_2026_10.csv")
print("\n=== Contract info (first 3 lines) ==="); head(f"{E}/CPSC_Contract_Info_2026_10.csv", 3)
print("\n=== Enrollment read-me ===")
print(open(f"{E}/Read_Me_CPSC_Enrollment_2026.txt", encoding="latin-1").read()[:3000])

In [0]:
%pip install pypdf --quiet

In [0]:
from pypdf import PdfReader
VOL = "/Volumes/workspace/default/raw_data/nb18"
r = PdfReader(f"{VOL}/crosswalk_2027/2027_plan_crosswalk_readme.pdf")
print("\n".join(page.extract_text() for page in r.pages)[:5000])

In [0]:
from pyspark.sql import functions as F

VOL = "/Volumes/workspace/default/raw_data/nb18"
E = f"{VOL}/enrollment_cpsc_2026_10/CPSC_Enrollment_2026_10"

xw = (spark.read.option("header", True).option("sep", "\t")
      .csv(f"{VOL}/crosswalk_2027/PlanCrosswalk2027_10012026.txt"))
enr = (spark.read.option("header", True).csv(f"{E}/CPSC_Enrollment_Info_2026_10.csv")
       .filter(F.col("State") == "MI"))
info = spark.read.option("header", True).csv(f"{E}/CPSC_Contract_Info_2026_10.csv")

print(f"Crosswalk rows: {xw.count():,}")
display(xw.groupBy("STATUS").count().orderBy(F.desc("count")))

print(f"Michigan enrollment rows: {enr.count():,} | suppressed (*): {enr.filter(F.col('Enrollment') == '*').count():,}")
mi_plans = enr.select(F.col("Contract Number").alias("Contract ID"), "Plan ID").distinct()
display(info.join(mi_plans, ["Contract ID", "Plan ID"])
        .groupBy("Organization Type", "EGHP").count().orderBy(F.desc("count")))

In [0]:
MA_TYPES = ["Local CCP", "Regional CCP", "PFFS", "MSA"]
ind = (info.filter(F.col("Organization Type").isin(MA_TYPES) & (F.col("EGHP") == "No"))
           .select("Contract ID", "Plan ID", "Organization Type", "Plan Type", "SNP Plan", "Parent Organization"))

xw26 = xw.select(F.col("PREVIOUS_CONTRACT_ID").alias("Contract ID"), F.col("PREVIOUS_PLAN_ID").alias("Plan ID"),
                 "STATUS", "CURRENT_CONTRACT_ID", "CURRENT_PLAN_ID")

# Guardrail 1: each 2026 plan must appear only once in the crosswalk, or the join would double-count members
dups = xw26.groupBy("Contract ID", "Plan ID").count().filter("count > 1").count()
print(f"2026 plans listed more than once in the crosswalk: {dups:,}")

mi = (enr.withColumnRenamed("Contract Number", "Contract ID")
         .join(ind, ["Contract ID", "Plan ID"])
         .withColumn("enr_low",  F.when(F.col("Enrollment") == "*", 0).otherwise(F.col("Enrollment").cast("int")))
         .withColumn("enr_high", F.when(F.col("Enrollment") == "*", 10).otherwise(F.col("Enrollment").cast("int"))))

# Guardrail 2: every non-suppressed count must be a clean number
bad = mi.filter(F.col("enr_low").isNull()).count()
assert bad == 0, f"{bad:,} enrollment values failed to convert to numbers"

mi_x = mi.join(xw26, ["Contract ID", "Plan ID"], "left")
mi_x.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.ma_mi_2026_plans_xw")

print(f"Michigan individual MA rows: {mi.count():,} | suppressed: {mi.filter(F.col('Enrollment') == '*').count():,}")
display(mi_x.groupBy(F.coalesce("STATUS", F.lit("NOT IN CROSSWALK")).alias("status"))
            .agg(F.countDistinct("Contract ID", "Plan ID").alias("plans"),
                 F.sum("enr_low").alias("members_low"),
                 F.sum("enr_high").alias("members_high"))
            .orderBy(F.desc("members_low")))

In [0]:
dup_keys = xw26.groupBy("Contract ID", "Plan ID").count().filter("count > 1")

combos = (xw26.join(dup_keys.select("Contract ID", "Plan ID"), ["Contract ID", "Plan ID"])
              .groupBy("Contract ID", "Plan ID")
              .agg(F.concat_ws(" + ", F.array_sort(F.collect_list("STATUS"))).alias("status_combo"),
                   F.count("*").alias("rows")))

display(combos.groupBy("status_combo")
              .agg(F.count("*").alias("plans"), F.max("rows").alias("max_rows_per_plan"))
              .orderBy(F.desc("plans")))

mi_dups = combos.join(mi.select("Contract ID", "Plan ID").distinct(), ["Contract ID", "Plan ID"]).count()
print(f"Duplicated plans that are Michigan individual MA plans: {mi_dups:,}")

In [0]:
def clean(df):
    for c in df.columns:
        df = df.withColumnRenamed(c, c.lower().replace(" ", "_"))
    return df

xw_agg = (xw26.filter(~F.col("STATUS").isin("New Plan", "Initial Contract"))
              .groupBy("Contract ID", "Plan ID")
              .agg(F.collect_set("STATUS").alias("statuses")))

T = "Terminated/Non-renewed Contract"
xw_agg = xw_agg.withColumn("outcome",
    F.when(F.array_contains("statuses", T) & (F.size("statuses") == 1), "Lost: plan ends")
     .when(F.array_contains("statuses", T), "Partial: ends in part of its area")
     .when(F.array_contains("statuses", "Renewal Plan with SAR"), "Shrinking: drops some counties")
     .when(F.array_contains("statuses", "Renewal Plan") | F.array_contains("statuses", "Renewal Plan with SAE"), "Continues")
     .when(F.array_contains("statuses", "Consolidated Renewal Plan"), "Merged into another plan")
     .otherwise("Other"))

assert xw_agg.groupBy("Contract ID", "Plan ID").count().filter("count > 1").count() == 0, "Still duplicated after grouping"

mi_x = (mi.join(xw_agg, ["Contract ID", "Plan ID"], "left")
          .withColumn("outcome", F.coalesce("outcome", F.lit("Not in crosswalk"))))
clean(mi_x.drop("statuses")).write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.ma_mi_2026_outcomes")

out = spark.table("workspace.default.ma_mi_2026_outcomes")
total_low = out.agg(F.sum("enr_low")).collect()[0][0]
print(f"Michigan individual MA rows: {out.count():,} | suppressed: {out.filter(F.col('enrollment') == '*').count():,}")
display(out.groupBy("outcome")
           .agg(F.countDistinct("contract_id", "plan_id").alias("plans"),
                F.sum("enr_low").alias("members_low"),
                F.sum("enr_high").alias("members_high"))
           .withColumn("share_of_members", F.round(F.col("members_low") / total_low * 100, 1))
           .orderBy(F.desc("members_low")))

In [0]:
out = spark.table("workspace.default.ma_mi_2026_outcomes")

cty = (out.groupBy("county")
          .agg(F.sum("enr_low").alias("all_members"),
               F.sum(F.when(F.col("outcome") == "Lost: plan ends", F.col("enr_low")).otherwise(0)).alias("lost_low"),
               F.sum(F.when(F.col("outcome") == "Lost: plan ends", F.col("enr_high")).otherwise(0)).alias("lost_high"))
          .withColumn("pct_lost", F.round(F.col("lost_low") / F.col("all_members") * 100, 1))
          .filter("all_members >= 1000"))

print("Most members losing their plan:")
display(cty.orderBy(F.desc("lost_low")).limit(10))
print("Highest share of members losing their plan:")
display(cty.orderBy(F.desc("pct_lost")).limit(10))

In [0]:
lost = out.filter(F.col("outcome") == "Lost: plan ends")
tot = lost.agg(F.sum("enr_low")).collect()[0][0]
rows = (lost.groupBy("parent_organization")
            .agg(F.sum("enr_low").alias("lost_low"), F.countDistinct("contract_id", "plan_id").alias("plans"))
            .orderBy(F.desc("lost_low")).collect())

print(f"Insurers with Michigan plans ending: {len(rows)}")
for i, r in enumerate(rows[:5], 1):
    print(f"  Insurer #{i}: {r['lost_low']:,} members ({r['lost_low'] / tot * 100:.0f}%), {r['plans']} plans")

west = ["Kent", "Ottawa", "Muskegon", "Allegan", "Kalamazoo", "Van Buren", "Newaygo", "Barry", "Montcalm", "Osceola"]
w = lost.filter(F.col("county").isin(west))
top = rows[0]["parent_organization"]
w_top = w.filter(F.col("parent_organization") == top).agg(F.sum("enr_low")).collect()[0][0] or 0
w_all = w.agg(F.sum("enr_low")).collect()[0][0]
print(f"\nWest Michigan members losing their plan: {w_all:,}, of whom {w_top:,} ({w_top / w_all * 100:.0f}%) are with Insurer #1")

In [0]:
# 1) What kind of plans are Insurer #1's two exits? (no names or IDs shown)
display(lost.filter(F.col("parent_organization") == top)
            .groupBy("plan_type", "snp_plan")
            .agg(F.sum("enr_low").alias("members_low"), F.countDistinct("county").alias("counties")))

# 2) When plans merge, do members stay with the same insurer?
parent = info.select(F.col("Contract ID").alias("cid"), F.col("Parent Organization").alias("parent")).distinct()
cons = (xw26.filter(F.col("STATUS") == "Consolidated Renewal Plan")
            .join(parent.withColumnRenamed("cid", "Contract ID").withColumnRenamed("parent", "prev_parent"), "Contract ID")
            .join(parent.withColumnRenamed("cid", "CURRENT_CONTRACT_ID").withColumnRenamed("parent", "curr_parent"),
                  "CURRENT_CONTRACT_ID", "left"))
display(cons.groupBy(F.when(F.col("curr_parent").isNull(), "2027 contract not in Oct 2026 file")
                      .when(F.col("prev_parent") == F.col("curr_parent"), "Same insurer")
                      .otherwise("Different insurer").alias("merge_destination")).count())

## Findings: Notebook 18

**Data:** CMS 2027 Part C&D Plan Crosswalk (Oct 1, 2026 release) and Monthly Enrollment by Contract/Plan/State/County, October 2026. Individual Medicare Advantage plans only (Local CCP, Regional CCP, PFFS, MSA; employer group, standalone drug, PACE, and cost plans excluded).

**Data quality decisions:**
- Enrollment counts of 10 or fewer are suppressed ("*"), covering 92% of all Michigan rows. Results are shown as a range (* = 0 to 10); ranges stayed narrow, so suppression has little effect on totals.
- 398 plans from 2026 appear more than once in the national crosswalk, likely because plan segments receive different outcomes (an assumption; the file has no segment field). Each 2026 plan was assigned one outcome by priority: Lost (every row terminated), Partial (terminated plus another status), Shrinking (service area reduction), Continues (renewal), Merged (consolidation only). A guardrail confirms no plan is counted twice.

**Michigan, 2026 to 2027 (about 968,000 individual MA members):**
- Continues: 58.5% | Shrinking (drops some counties): 24.2% | Merged into another plan: 13.9% | **Lost (plan ends): 3.4%, about 32,700 to 35,100 members** | Partial: about 0
- More than 4 in 10 members (41.5%) are in a plan that ends, merges, or shrinks.
- Every Michigan plan matched the crosswalk.

**Concentration:** 19 insurers have Michigan plans ending, but one insurer's two plans account for **82%** of members losing their plan (about 27,000; 87% of them in a PPO). In West Michigan, the same insurer accounts for **96%** of the loss.

**Geography:** Kent County has the most members losing their plan (6,882; 9.3% of its members), followed by Ottawa (3,460; 8.9%). Wayne is third by count but only 1.8%. Nine of the ten highest-share counties are in West or west-central Michigan; Midland is highest at 10.3%.

**Merges stay in-house:** In all 805 consolidations where the 2027 insurer could be identified, members stay with the same insurer.

**What I'd tell a COO:** About 33,000 to 35,000 Michigan Medicare Advantage members must choose new coverage for January, and most come from one insurer's two plans in West Michigan, mostly PPO members who may move into narrower networks. Health systems and DME suppliers should expect re-authorizations, network checks, and supplier changes to land together in January, heaviest on the west side of the state.

**What this data can't answer:** Which counties the shrinking plans drop; where displaced members actually enroll; why the insurer ended its plans; and segment-level outcomes within a plan.

**What I'd need next:** 2027 service area or landscape files for the counties being dropped; January and February 2027 enrollment to see where displaced members went; and plan announcements or public filings for the reasons behind the exits.

**Publishing rule for this notebook:** Insurers are described by plan type and region, not named. Outcomes are reported by county and plan category.